<a href="https://colab.research.google.com/github/Saritha-batch1/Multi-Model-AI-Agent/blob/main/multimodel_on_HD_Deekshitha_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("/content/kidney_disease.csv")
display(df.head())

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [27]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [28]:
# Shape & basic information
print("Shape of dataset:", df.shape)
df.info()

# Check missing values
df.isnull().sum()




Shape of dataset: (400, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              400 non-null    int64  
 1   age             391 non-null    float64
 2   bp              388 non-null    float64
 3   sg              353 non-null    float64
 4   al              354 non-null    float64
 5   su              351 non-null    float64
 6   rbc             248 non-null    object 
 7   pc              335 non-null    object 
 8   pcc             396 non-null    object 
 9   ba              396 non-null    object 
 10  bgr             356 non-null    float64
 11  bu              381 non-null    float64
 12  sc              383 non-null    float64
 13  sod             313 non-null    float64
 14  pot             312 non-null    float64
 15  hemo            348 non-null    float64
 16  pcv             330 non-null    object 
 17  wc     

,0
id,0
age,9
bp,12
sg,47
al,46
su,49
rbc,152
pc,65
pcc,4
ba,4


In [29]:
# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.head()


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [30]:
df.columns


Index(['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr',
       'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')

In [31]:
# Detect target column automatically
possible_targets = ["class", "classification", "Class", "target", "outcome", "Stage"]

target_col = None
for col in df.columns:
    if col in possible_targets:
        target_col = col
        break

if target_col is None:
    raise ValueError(" NO target column found. Check df.columns manually!")

print("Target Column Detected:", target_col)


Target Column Detected: classification


In [32]:
X = df.drop(target_col, axis=1)
y = df[target_col]


In [33]:


# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Replace ? with NaN
df = df.replace("?", np.nan)

# Detect numeric columns
numeric_cols = ["age","bp","bgr","bu","sc","sod","pot","hemo","pcv","wc","rc"]
numeric_cols = [c for c in numeric_cols if c in df.columns]

# Convert numeric
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill numeric NaN
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Detect categorical columns
cat_cols = [c for c in df.columns if df[c].dtype == "object"]

# Fill categorical NaN
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])



# Encode categorical
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# Split
X = df.drop(target_col, axis=1)
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (400, 25)
y shape: (400,)


In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


Training shape: (320, 25)
Testing shape: (80, 25)


In [35]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)


RandomForestClassifier(n_estimators=200, random_state=42)

In [36]:
y_pred = model.predict(X_test)


In [37]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 1.0

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        52
           2       1.00      1.00      1.00        28

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80



Mile stone -3 -Finding Synthesis
rebuilt the analytical logic using dataset-specific clinical parameters, generated explainable risk scores, synthesized findings, and produced personalized health recommendations aligned with identified risks

In [38]:
print(df.columns.tolist())


['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


In [39]:
import pandas as pd
import numpy as np

numeric_cols = [
    "bgr","bu","sc","sod","pot",
    "hemo","pcv","wc","rc"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())


In [40]:
THRESHOLDS = {
    "sc_high": 1.5,
    "bu_high": 40,
    "hemo_low": 12,
    "pcv_low": 36,
    "bgr_high": 140
}


In [41]:
def kidney_risk(row):
    if row["sc"] > THRESHOLDS["sc_high"] and row["bu"] > THRESHOLDS["bu_high"]:
        return 2
    elif row["sc"] > THRESHOLDS["sc_high"]:
        return 1
    else:
        return 0

df["kidney_risk"] = df.apply(kidney_risk, axis=1)


In [42]:
def anemia_risk(row):
    if row["hemo"] < THRESHOLDS["hemo_low"] and row["pcv"] < THRESHOLDS["pcv_low"]:
        return 2
    elif row["hemo"] < THRESHOLDS["hemo_low"]:
        return 1
    else:
        return 0

df["anemia_risk"] = df.apply(anemia_risk, axis=1)


In [43]:
def anemia_risk(row):
    if row["hemo"] < THRESHOLDS["hemo_low"] and row["pcv"] < THRESHOLDS["pcv_low"]:
        return 2
    elif row["hemo"] < THRESHOLDS["hemo_low"]:
        return 1
    else:
        return 0

df["anemia_risk"] = df.apply(anemia_risk, axis=1)


In [44]:
def metabolic_risk(row):
    if row["bgr"] > THRESHOLDS["bgr_high"] and row["dm"] == 1:
        return 2
    elif row["bgr"] > THRESHOLDS["bgr_high"]:
        return 1
    else:
        return 0

df["metabolic_risk"] = df.apply(metabolic_risk, axis=1)


In [45]:
df["total_risk_score"] = (
    df["kidney_risk"] +
    df["anemia_risk"] +
    df["metabolic_risk"]
)

def risk_category(score):
    if score >= 4:
        return "High Risk"
    elif score >= 2:
        return "Moderate Risk"
    else:
        return "Low Risk"

df["risk_category"] = df["total_risk_score"].apply(risk_category)


In [46]:
def synthesize_findings(row):
    findings = []

    if row["kidney_risk"] == 2:
        findings.append("Severe kidney function impairment")
    elif row["kidney_risk"] == 1:
        findings.append("Mild kidney dysfunction")

    if row["anemia_risk"] == 2:
        findings.append("Severe anemia detected")
    elif row["anemia_risk"] == 1:
        findings.append("Mild anemia detected")

    if row["metabolic_risk"] >= 1:
        findings.append("Metabolic imbalance observed")

    if not findings:
        findings.append("No major abnormalities detected")

    return "; ".join(findings)

df["synthesized_findings"] = df.apply(synthesize_findings, axis=1)


In [47]:
def generate_recommendations(row):
    recs = []

    if row["kidney_risk"] >= 1:
        recs.append("Reduce salt intake and monitor kidney parameters")

    if row["anemia_risk"] >= 1:
        recs.append("Increase iron-rich foods and consult physician")

    if row["metabolic_risk"] >= 1:
        recs.append("Control blood glucose with diet and exercise")

    if row["risk_category"] == "High Risk":
        recs.append("Immediate medical consultation recommended")

    if not recs:
        recs.append("Maintain healthy lifestyle")

    return " | ".join(recs)

df["personalized_recommendations"] = df.apply(generate_recommendations, axis=1)


In [48]:
df[[
    "risk_category",
    "synthesized_findings",
    "personalized_recommendations"
]].head()


,risk_category,synthesized_findings,personalized_recommendations
0,Low Risk,No major abnormalities detected,Maintain healthy lifestyle
1,Low Risk,Mild anemia detected,Increase iron-rich foods and consult physician
2,High Risk,Severe kidney function impairment; Severe anem...,Reduce salt intake and monitor kidney paramete...
3,High Risk,Severe kidney function impairment; Severe anem...,Reduce salt intake and monitor kidney paramete...
4,Moderate Risk,Severe anemia detected,Increase iron-rich foods and consult physician


MULTI-MODEL ORCHESTRATOR

In [49]:
def multi_model_orchestrator(row):
    return {
        "kidney_risk": row["kidney_risk"],
        "anemia_risk": row["anemia_risk"],
        "metabolic_risk": row["metabolic_risk"],
        "risk_category": row["risk_category"],
        "findings": row["synthesized_findings"],
        "recommendations": row["personalized_recommendations"]
    }


In [50]:
df["orchestrated_output"] = df.apply(multi_model_orchestrator, axis=1)


REPORT GENERATION MODULE

In [51]:
def generate_report(row, patient_id):
    report = f"""
    ===============================
    AI-BASED HEALTH DIAGNOSTIC REPORT
    ===============================

    Patient ID: {patient_id}

    Overall Risk Category:
    → {row['risk_category']}

    Key Findings:
    → {row['synthesized_findings']}

    Personalized Recommendations:
    → {row['personalized_recommendations']}

    Medical Disclaimer:
    This report is generated using an AI-based analytical system and is intended
    for preliminary health assessment only. It should not be considered a
    substitute for professional medical diagnosis or treatment.

    ===============================
    """
    return report


In [52]:
final_reports = []

for idx, row in df.iterrows():
    try:
        report = generate_report(row, patient_id=idx)
        final_reports.append(report)
    except Exception as e:
        final_reports.append(f"Error processing patient {idx}: {e}")


In [53]:
print(final_reports[0])



    AI-BASED HEALTH DIAGNOSTIC REPORT

    Patient ID: 0

    Overall Risk Category:
    → Low Risk

    Key Findings:
    → No major abnormalities detected

    Personalized Recommendations:
    → Maintain healthy lifestyle

    Medical Disclaimer:
    This report is generated using an AI-based analytical system and is intended
    for preliminary health assessment only. It should not be considered a
    substitute for professional medical diagnosis or treatment.

    


ERROR HANDLING & EDGE CASE MANAGEMENT

In [55]:
def safe_generate_report(row, patient_id):
    try:
        return generate_report(row, patient_id)
    except:
        return f"Report could not be generated for Patient {patient_id} due to missing or invalid data."


In [56]:
df["final_report"] = [
    safe_generate_report(row, idx)
    for idx, row in df.iterrows()
]


EXPORT FINAL REPORTS

In [57]:
with open("final_health_reports.txt", "w") as f:
    for report in df["final_report"]:
        f.write(report + "\n\n")


For Milestone-4, I integrated all the models into one complete workflow. The system analyzes the data, summarizes the findings, and generates an easy-to-understand health report with recommendations. I also tested the full pipeline and added error handling to ensure reliable output. The final solution is explainable, practical, and clinically meaningful

Thus the project produces an automated, explainable health diagnostic report that summarizes patient risk levels and provides personalized recommendations based on blood test analysis.